In [2]:
import torch.nn as nn
import torch
import math


In [3]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention module"""
    def __init__(self,num_heads, d_model, dropout =0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.dropout = dropout
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)

        self.W_o = nn.Linear(d_model,d_model)

    def concatenate(self,x):
        x = x.transpose(1, 2)
        x = x.contiguous().view(x.size(0), -1, self.d_model)
        return x

    def forward(self,x,mask=None):
        batch_size = x.size(0)

        Q= self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape output size to : (batch_size,seq_len,num_heads,head_dim)
        Q=Q.reshape(batch_size,-1,self.num_heads,self.head_dim)
        K=K.reshape(batch_size,-1,self.num_heads,self.head_dim)
        V=V.reshape(batch_size,-1,self.num_heads,self.head_dim)

        #Permute seq_len and num_heads
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        #calculate attention score
        scores = torch.matmul(
            Q,
            K.transpose(2,3)
        )/math.sqrt(self.head_dim)
        if mask is not None:
            # Add .unsqueeze(1).unsqueeze(1) to shape the mask correctly
            # New mask shape: (Batch, 1, 1, Seq_Len)
            # This allows it to broadcast to (Batch, Heads, Seq_Len, Seq_Len)
            scores = scores.masked_fill_(mask.unsqueeze(1).unsqueeze(1) == 0, -1e9)
        #Softmax
        scores = torch.softmax(scores, dim=-1)
        #Determine attention
        attention = torch.matmul(scores,V)
        attention=torch.dropout(attention,p=self.dropout,train=True)

        #Concatenate the heads
        attention = self.concatenate(attention)
        return self.W_o(attention) # Output shape: (batch_size,seq_len,d_model)

In [4]:
class PositionalEncoding(nn.Module):
    """Positional Encoding module"""
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        self.max_len = max_len

        # 1. Initialize as 2D: (max_len, d_model)
        P = torch.zeros(max_len, d_model)

        numerator = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1)
        denominator = torch.pow(10000, torch.arange(0, d_model, 2, dtype=torch.float32) / d_model)
        X = numerator / denominator

        # 2. Use 2D indexing
        P[:, 0::2] = torch.sin(X)
        P[:, 1::2] = torch.cos(X)

        # 3. Unsqueeze ONCE to make it 3D: (1, max_len, d_model)
        self.register_buffer('P', P.unsqueeze(0))

    def forward(self, X):
        # X is (Batch, Seq_Len, d_model)
        # self.P is (1, Max_Len, d_model)
        # Slicing self.P results in (1, Seq_Len, d_model)
        X = X + self.P[:, :X.shape[1], :]
        return self.dropout(X)


In [5]:
class AddNormLay(nn.Module):
    """Residual Connection + Normalization Layer"""
    def __init__(self,norm_shape,dropout=0.1):
        super().__init__()
        self.LayerNorm = nn.LayerNorm(norm_shape)
        self.dp =nn.Dropout(p=dropout)
    def forward(self,initial,x):
        return self.LayerNorm(self.dp(x) +initial) # output shape: (batch_size, seq_len,d_model)

In [6]:
class FFN(nn.Module):
    """Feedforward Neural Network"""
    def __init__(self,ffn_num_input,ffn_num_hidden):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(ffn_num_input,ffn_num_hidden),
            nn.ReLU(),
            nn.Linear(ffn_num_hidden,ffn_num_input),
        )
    def forward(self,x):
        return self.ffn(x)

In [7]:
class EncoderBlock(nn.Module):
    """Encoder Block"""
    def __init__(self,num_heads,d_model):
        super().__init__()
        self.m_attention = MultiHeadAttention(num_heads=num_heads,d_model=d_model,dropout=0.1)
        self.norm_layer_1 = AddNormLay(d_model,dropout=0.1)
        self.norm_layer_2 = AddNormLay(d_model,dropout=0.1)
        self.ffn = FFN(d_model,d_model*4)

    def forward(self,x,mask=None):
        attention_output = self.m_attention(x,mask)
        x = self.norm_layer_1(x,attention_output)
        return self.norm_layer_2(x,self.ffn(x))

In [8]:
class TransformerEncoder(nn.Module):
    """Transformer Encoder"""
    def __init__(self,max_len,num_heads,d_model,num_blk,vocab_size):
        super().__init__()
        self.d_model = d_model
        self.positional_encoding = PositionalEncoding(d_model,dropout=0.1,max_len=max_len)
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.blks = nn.Sequential()
        for _ in range(num_blk):
            self.blks.add_module("EncoderBlock"+str(_),EncoderBlock(num_heads,d_model))

    def forward(self,x,mask=None):
        x = self.positional_encoding(self.embedding(x) * math.sqrt(self.d_model))
        for i,blk in enumerate(self.blks):
            x = blk(x,mask)
        return x

In [9]:
"""Mock Classification Test"""
vocabulary_size = 10000
seq_len = 100
dimension = 512
blk_numb = 8
heads_num = 8
batch_size = 32

class SimpleClassifier(nn.Module):
    """Simple Classifier Module"""
    def __init__(self,d_model):
        super().__init__()
        self.fc = nn.Linear(d_model,1)
        self.sigmoid = nn.Sigmoid()
    def forward(self,x):
        first_token_out = x[:, 0, :]
        out = self.fc(first_token_out)
        return self.sigmoid(out)

encoder_model = nn.Sequential(
    TransformerEncoder(max_len=seq_len,num_heads=heads_num,d_model=dimension,num_blk=blk_numb,vocab_size=vocabulary_size)
    ,
    SimpleClassifier(d_model=dimension)
)

#Create dummy data
dummy_input = torch.randint(1,vocabulary_size,(batch_size,seq_len))
#Create dummy target
dummy_targets = torch.randint(0,2,(batch_size,1)).float()

print("Testing Forward Pass...")
outputs = encoder_model(dummy_input)
print(f"Input Shape: {dummy_input.shape}")
print(f"Final Prediction Shape: {outputs.shape}") # Should be (32, 1)
print("Forward pass successful!\n")

# --- 6. Test Backward Pass (Training Step) ---
print("Testing Backward Pass (Learning)...")
criterion = nn.BCELoss() # Binary Cross Entropy
optimizer = torch.optim.Adam(encoder_model.parameters(), lr=0.001)

optimizer.zero_grad()
loss = criterion(outputs, dummy_targets)
loss.backward()
optimizer.step()

print(f"Loss: {loss.item():.4f}")
print("Backward pass successful! Your Encoder is working.")

Testing Forward Pass...
Input Shape: torch.Size([32, 100])
Final Prediction Shape: torch.Size([32, 1])
Forward pass successful!

Testing Backward Pass (Learning)...
Loss: 0.6877
Backward pass successful! Your Encoder is working.


In [10]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
"""Training With real Data"""
data =[
    ("I love this film",1),
    ("This is so boring",0),
    ("This is great",1),
    ("I can't watch it again",0),
    ("Masterpiece",1),
    ("Boring and slow",0)
]

vocab = {"<PAD>":0,"<UNK>":1}

for text,label in data:
    words=text.lower().split()
    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)

print("Vocabulary: "+str(vocab))

def tokenize(text,vocab):
    return [vocab.get(word,vocab["<UNK>"]) for word in text.lower().split()]

class TextDataset(Dataset):
    def __init__(self,data,vocab):
        self.data = data
        self.vocab = vocab
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        text, label = self.data[idx]
        token_ids = tokenize(text,vocab)
        return torch.tensor(token_ids,dtype=torch.long), torch.tensor(label,dtype=torch.float)
def collate_fn(batch):
    input,labels = zip(*batch)
    pad_inputs = pad_sequence(input, batch_first=True, padding_value=0)

    mask = (pad_inputs !=0).long()
    labels = torch.stack(labels)
    return pad_inputs, mask,labels

dataset = TextDataset(data,vocab)
dataloader = DataLoader(dataset,batch_size=2, collate_fn=collate_fn,shuffle=True)

new_vocab_size = len(vocab)
new_num_heads = 4
new_num_dim = 32
new_seq_len=8
encoder = TransformerEncoder(new_seq_len,new_num_heads,new_num_dim,2,new_vocab_size)
classifier = SimpleClassifier(new_num_dim)
optimizer = torch.optim.Adam(list(encoder.parameters())+list(classifier.parameters()), lr=0.001)
criterion = nn.BCELoss()
encoder.train()
classifier.train()

for epoch in range(5):
    train_loss = 0
    for batch_x, mask, batch_y in dataloader:
        features = encoder(batch_x,mask)
        predictions = classifier(features)
        optimizer.zero_grad()
        loss = criterion(predictions, batch_y.unsqueeze(1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}')




Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'this': 4, 'film': 5, 'is': 6, 'so': 7, 'boring': 8, 'great': 9, "can't": 10, 'watch': 11, 'it': 12, 'again': 13, 'masterpiece': 14, 'and': 15, 'slow': 16}
Epoch 1, Train Loss: 2.2050
Epoch 2, Train Loss: 1.8664
Epoch 3, Train Loss: 1.7890
Epoch 4, Train Loss: 1.5984
Epoch 5, Train Loss: 1.3794


In [13]:
encoder.eval()
classifier.eval()

my_review ="Disappointing"
tokens = torch.tensor(tokenize(my_review,vocab),dtype=torch.long)
tokens = tokens.unsqueeze(0)
mask = (tokens !=0).long()

with torch.no_grad():
    feat = encoder(tokens,mask)
    predict = classifier(feat)
    output = predict.item()
    if output>0.6:
        print("Review Positive")
    else:
        print("Review Negative")

Review Negative
